# Prepare the dataset

In [ ]:
import cv2
import os
import re
import pandas as pd
import albumentations as A
from pathlib import Path

In [ ]:
IMG_DIR = Path("../0_datasets/labelled_images/images")   
LBL_DIR = Path("../0_datasets/labelled_images/labels")   
SAVE_DIR = Path("OCR_Dataset")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
AUG_COUNT_PER_IMAGE = 5 

transform = A.Compose([
    A.OneOf([
        A.RandomBrightnessContrast(p=1),
        A.HueSaturationValue(p=1),
    ], p=0.5),
    A.OneOf([
        A.Perspective(scale=(0.02, 0.05), p=1), # Angle variation
        A.Rotate(limit=5, border_mode=cv2.BORDER_CONSTANT, p=1),
    ], p=0.4),
    A.OneOf([
        A.MotionBlur(blur_limit=3, p=1),
        A.GaussNoise(var_limit=(10.0, 30.0), p=1),
    ], p=0.3),
    A.Sharpen(alpha=(0.1, 0.3), p=0.2), # Simulates edge sharpening
])

In [ ]:
def extract_valid_label(filename):
    fname_upper = filename.upper()
    
    # 1. Special Case: Water Room
    if "WATER_ROOM" in fname_upper:
        return "Water_Room"
    
    # 2. Special Case: MC102A
    if "MC102A" in fname_upper:
        return "MC102A"
    
    # 3. Pattern: MA followed by 3 digits
    match = re.search(r"MA\d{3}", fname_upper)
    if match:
        return match.group()
    
    return None

In [ ]:
data_rows = []
processed_count = 0

all_labels = list(LBL_DIR.glob("*.txt"))

for label_file in all_labels:
    gt_label = extract_valid_label(label_file.stem)
    if not gt_label: continue 

    # --- CASE-INSENSITIVE IMAGE FINDER ---
    img_path = None
    # Check for every possible common extension
    for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
        temp = IMG_DIR / (label_file.stem + ext)
        if temp.exists():
            img_path = temp
            break
            
    if img_path is None:
        print(f"Image not found for {label_file.name} (checked .jpg, .JPG, .png, etc.)")
        continue
        
    img = cv2.imread(str(img_path))
    if img is None:
        print(f"Could not read image {img_path}")
        continue
        
    h_img, w_img, _ = img.shape

    with open(label_file, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
    
    if not lines: continue

    # Take the first available box
    first_line = lines[0].split()
    if len(first_line) < 5: continue
        
    try:
        coords = [float(x) for x in first_line[1:5]]
        x_c, y_c, w, h = coords
        
        # Convert to pixels
        x1, y1 = int((x_c - w/2) * w_img), int((y_c - h/2) * h_img)
        x2, y2 = int((x_c + w/2) * w_img), int((y_c + h/2) * h_img)
        
        # Crop with boundary safety
        raw_crop = img[max(0, y1):min(h_img, y2), max(0, x1):min(w_img, x2)]
        
        if raw_crop is None or raw_crop.size == 0:
            print(f"Empty crop for {label_file.stem} (Check label coordinates)")
            continue

        base_name = f"id{processed_count:04d}_{gt_label}"

        # 1. Save Original
        orig_name = f"{base_name}_orig.jpg"
        cv2.imwrite(str(SAVE_DIR / orig_name), raw_crop)
        data_rows.append({"file_name": orig_name, "ground_truth": gt_label})

        # 2. Save Augmented versions
        for aug_i in range(AUG_COUNT_PER_IMAGE):
            augmented = transform(image=raw_crop)["image"]
            aug_name = f"{base_name}_aug{aug_i}.jpg"
            cv2.imwrite(str(SAVE_DIR / aug_name), augmented)
            data_rows.append({"file_name": aug_name, "ground_truth": gt_label})
            
        processed_count += 1
        
    except Exception as e:
        print(f"Unexpected error on {label_file.stem}: {e}")
        continue

if data_rows:
    df = pd.DataFrame(data_rows)
    df.to_csv(SAVE_DIR / "labels.csv", index=False)
    print(f"\nDataset Built! Processed {processed_count} images.")
    print(f"Total entries in labels.csv: {len(df)}")

# Train each model on our dataset

In [ ]:
import os
import io
import json
import shutil
import subprocess
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
BASE_DIR       = "./OCR_Dataset"
IMAGE_DIR  = os.path.join(BASE_DIR)
CSV_PATH   = os.path.join(BASE_DIR, "labels.csv")

PADDLE_OUT = "./outputs/paddleocr"
EASYOCR_OUT= "./outputs/easyocr"
TESS_OUT   = "./outputs/tesseract"

for d in [PADDLE_OUT, EASYOCR_OUT, TESS_OUT]:
    os.makedirs(d, exist_ok=True)

print("Paths configured:")
print(f"  Images  : {IMAGE_DIR}")
print(f"  CSV     : {CSV_PATH}")
print(f"  PaddleOCR output : {PADDLE_OUT}")
print(f"  EasyOCR output   : {EASYOCR_OUT}")
print(f"  Tesseract output : {TESS_OUT}")

In [ ]:
df = pd.read_csv(CSV_PATH)

# Validate expected columns
assert 'file_name' in df.columns and 'ground_truth' in df.columns, \
    "CSV must have 'file_name' and 'ground_truth' columns!"

# Check which images actually exist on disk
df['img_path'] = df['file_name'].apply(lambda f: os.path.join(IMAGE_DIR, f))
df['exists']   = df['img_path'].apply(os.path.exists)

matched_df    = df[df['exists']].reset_index(drop=True)
missing_df    = df[~df['exists']].reset_index(drop=True)

print("=" * 45)
print("        DATASET INTEGRITY CHECK")
print("=" * 45)
print(f"  Total rows in CSV         : {len(df)}")
print(f"  Images found on disk      : {len(matched_df)}")
print(f"  Images missing on disk   : {len(missing_df)}")
print("=" * 45)

if len(missing_df) > 0:
    print("\nMissing files (first 5):")
    print(missing_df['file_name'].head().tolist())

print("\nSample CSV entries:")
print(matched_df[['file_name', 'ground_truth']].head())

In [ ]:
samples = matched_df.sample(min(4, len(matched_df)), random_state=42)

fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
if len(samples) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, samples.iterrows()):
    img = mpimg.imread(row['img_path'])
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
    ax.set_title(f'"{row["ground_truth"]}"', fontsize=10, wrap=True)
    ax.axis("off")

plt.suptitle("Dataset Preview", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

SEED      = 42
VAL_RATIO = 0.2

train_df, val_df = train_test_split(
    matched_df,
    test_size=SEED,
    random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# Keep 'matched' as a list of stems for backward compatibility with training cells
train_set = train_df['file_name'].apply(lambda f: Path(f).stem).tolist()
val_set   = val_df['file_name'].apply(lambda f: Path(f).stem).tolist()

print(f"  Total samples : {len(matched_df)}")
print(f"  Train split   : {len(train_df)} ({int((1-VAL_RATIO)*100)}%)")
print(f"  Val split     : {len(val_df)}   ({int(VAL_RATIO*100)}%)")